# Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

- Tracking agent behavior with logging, analytics, and debugging.
- Transforming prompts, tool selection, and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guardrails, and PII detection.


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(dotenv_path=r"config\.env")

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_GENAI_API_KEY")
GOOGLE_MODEL = os.getenv("GOOGLE_MODEL")
GROQ_MODEL = os.getenv("GROQ_MODEL")

## Summarization MiddleWare

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

In [4]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import SystemMessage, HumanMessage

### Messagebased summarization middleware
agent = create_agent(
    model=GOOGLE_MODEL,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=GOOGLE_MODEL,
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [5]:
### Run with thread id
config={"configurable":{"thread_id":"test-1"}}

In [6]:
# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='e5402b13-5963-413d-844c-63037342c3ee'), AIMessage(content=[{'type': 'text', 'text': '2 + 2 = 4', 'extras': {'signature': 'El4KXAERTTIPWbxJX5bJbP5ATqWax/sziIsTFGNAzx9WbTmEsP+AHbIQ3htj3KXaeGt3zaLaWosYRMwume68S9tzmzGj0l43mYjEsohTVGEdva6uO3Won4hWI7XpSk/q'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a02bb4-d5df-7fc0-9228-fa2d95093095-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 7, 'total_tokens': 15, 'input_token_details': {'cache_read': 0}})]}
Messages: 2
Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='e5402b13-5963-413d-844c-63037342c3ee'), AIMessage(content=[{'type': 'text', 'text': '2 + 2 = 4', 'extras': {'signature': 'El4KXAERTTIPWbxJX5b

## TokenSize

In [7]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens"""
    return f"""Found hotels in {city}
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""

agent = create_agent(
    model=GOOGLE_MODEL,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=GOOGLE_MODEL,
            trigger=("tokens",550),
            keep=("tokens",200)
        )
    ]
)

config={"configurable":{"thread_id":"test-1"}}

# token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # Approximate token count (assuming 4 characters per token)

In [8]:
# Run test
cities = ["New York", "Los Angeles", "Chicago", "Houston", "Phoenix"]

for city in cities:
    response=agent.invoke(
        {"messages":[HumanMessage(content=f"Search hotels in {city}")]},
        config=config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

New York: ~153 tokens, 4 messages
[HumanMessage(content='Search hotels in New York', additional_kwargs={}, response_metadata={}, id='eb8441b0-7c43-4b65-8dde-44a690e72593'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_hotels', 'arguments': '{"city": "New York"}'}, '__gemini_function_call_thought_signatures__': {'call_1922300': 'El4KXAERTTIP+m/125Ov3vakIt84bHO6irPAU4cGzrk8vd91QsF7WH4z/1bPgbUPfO+83oQkriizRuGPdzh46NVCx5FYqPiR4bVhH7y3VNts8J6esXBNb8HQ5IX7slTq'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a02bbd-eeb0-7782-95fc-1582077a721e-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'New York'}, 'id': 'call_1922300', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 53, 'output_tokens': 17, 'total_tokens': 70, 'input_token_details': {'cache_read': 0}}), ToolMessage(content='Found hotels in New York\n    1. Gra

## Fraction

In [9]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens"""
    return f"""Found hotels in {city}
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""

agent = create_agent(
    model=GOOGLE_MODEL,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=GOOGLE_MODEL,
            trigger=("fraction",0.005), # 0.5% = ~640 tokens
            keep=("fraction",0.002)     # 0.2% = ~256 tokens
        )
    ]
)

config={"configurable":{"thread_id":"test-1"}}

# token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # Approximate token count (assuming 4 characters per token)

# Run test
cities = ["New York", "Los Angeles", "Chicago", "Houston", "Phoenix"]

for city in cities:
    response=agent.invoke(
        {"messages":[HumanMessage(content=f"Search hotels in {city}")]},
        config=config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

New York: ~160 tokens, 4 messages
[HumanMessage(content='Search hotels in New York', additional_kwargs={}, response_metadata={}, id='d43c3a9f-ca4d-4208-9129-b0c26f3eb30c'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_hotels', 'arguments': '{"city": "New York"}'}, '__gemini_function_call_thought_signatures__': {'call_1350694': 'El4KXAERTTIPo0bN8UPKzxSfP0WDdP1sZhM0d5qfD9kgQji/NjLSNaf2cxshd3G0GyWwNea1Qr77LIdxNYnwZ5LTIjl6Tlx3sfgW1AcwpHUTlEnVBr0l8yn7LyDXlyl0'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a02bbf-ca68-7301-8718-7d5cc3faa4ab-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'New York'}, 'id': 'call_1350694', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 53, 'output_tokens': 17, 'total_tokens': 70, 'input_token_details': {'cache_read': 0}}), ToolMessage(content='Found hotels in New York\n    1. Gra


## Human In the Loop MiddleWare

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

- High-stakes operations requiring human approval (e.g. database writes, financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent=create_agent(
    model=GOOGLE_MODEL,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,

            }
        )
    ]
)

config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [12]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='6459eeaa-3eb1-45a0-9125-433931a2d421'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"subject": "Hello", "recipient": "john@test.com", "body": "How are you?"}'}, '__gemini_function_call_thought_signatures__': {'call_2706511': 'El4KXAERTTIPvebnkkvILw5CXhfo6d8jgpZBYHve9ehuSvUWIf+mB8WZ41iy9klHrb2k6otJOYtpv52RwWKhPSu5IjMai6wXhByWdnv2cCVeXKQZkvtWBen8cpoycFXp'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a02bc1-7491-7583-ac91-0617a54a0180-0', tool_calls=[{'name': 'send_email_tool', 'args': {'subject': 'Hello', 'recipient': 'john@test.com', 'body': 'How are you?'}, 'id': 'call_2706511', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 144, '

In [13]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")



⏸️ Paused! Approving...
✅ Result: [{'type': 'text', 'text': "Email has been successfully sent to john@test.com with the subject 'Hello'.", 'extras': {'signature': 'El4KXAERTTIPDVRG+jFxU4uJ0fvO/atGOVCw7P0snqZrKTaqQqM4uJQegK7dzAg0o7gsdUD9nLjvsox+U1LIFDefI4qn3i7ssHt8qr+CUTZTJ+SBMufBmhbsWi1Kaon+'}}]


In [14]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='6459eeaa-3eb1-45a0-9125-433931a2d421'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"subject": "Hello", "recipient": "john@test.com", "body": "How are you?"}'}, '__gemini_function_call_thought_signatures__': {'call_2706511': 'El4KXAERTTIPvebnkkvILw5CXhfo6d8jgpZBYHve9ehuSvUWIf+mB8WZ41iy9klHrb2k6otJOYtpv52RwWKhPSu5IjMai6wXhByWdnv2cCVeXKQZkvtWBen8cpoycFXp'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a02bc1-7491-7583-ac91-0617a54a0180-0', tool_calls=[{'name': 'send_email_tool', 'args': {'subject': 'Hello', 'recipient': 'john@test.com', 'body': 'How are you?'}, 'id': 'call_2706511', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 144, '

## Reject

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model=GOOGLE_MODEL,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)



In [17]:
config = {"configurable": {"thread_id": "test-reject"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config)

In [18]:
# Step 2: Reject
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: [{'type': 'text', 'text': 'The email could not be sent because the tool call was rejected by the user. Let me know if you need help with anything else!', 'extras': {'signature': 'El4KXAERTTIPeyub5bBXxxZ8gT1s5kteB8LAfjWtpeOmgM4e66JAdpPedua3W+ZpmguaPO/7M4HwukuAM4FeQU3JAHKlp5jqiJv1kynXiy7V4BJv1mbQ/02m9BmydE1F'}}]


In [19]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='cbf3909a-12e0-4bc1-b608-7f570c4c6679'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"body": "How are you?", "subject": "Hello", "recipient": "john@test.com"}'}, '__gemini_function_call_thought_signatures__': {'call_754329': 'El4KXAERTTIPoV2HhMr1SY/qrIjkzjN3atvQw8Hl+jXemAaB6jWzEbqd1gK68jsfVu9mLhuNCXff2x5RYv9CdXDwHr5Dl2oWBz1A4df21uhwPn5R/keuDnw9TQlbpMT3'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a02bc4-f8c8-74f3-8020-28ce1d667d32-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'How are you?', 'subject': 'Hello', 'recipient': 'john@test.com'}, 'id': 'call_754329', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 144, 'ou

## Editing

In [20]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model=GOOGLE_MODEL,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [21]:
config = {"configurable": {"thread_id": "test-edit"}}

# Step 1: Request (with wrong info)
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'")]},
    config=config
)

result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='7f9b352e-a06e-483c-a364-5795f1cc9a6a'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"body": "Hello", "subject": "Test", "recipient": "wrong@email.com"}'}, '__gemini_function_call_thought_signatures__': {'call_2433651': 'El4KXAERTTIP3ZPPc2ylW/PTDVcllWBk95AKLBmsIZiuteST8SsGPHGohuxrVtWSQRdndgRHKd0vIF5P4I/QYg41UeQllwCMm84jH18cJX8//WWirYl8FAHNafqb/Rou'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a02bc6-61de-7702-b31f-68c6eb7f7dce-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'Hello', 'subject': 'Test', 'recipient': 'wrong@email.com'}, 'id': 'call_2433651', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 142, 'output_tokens': 34

In [22]:
# Step 2: Edit and approve
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                   # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )
    
    print(f"✏️ Result: {result['messages'][-1].content}")

⏸️ Paused! Editing...
✏️ Result: [{'type': 'text', 'text': "I have successfully sent an email to correct@email.com with the subject 'Corrected Subject' and the body 'This was edited by human before sending'.", 'extras': {'signature': 'El4KXAERTTIP3AxoCbT7RyU83aT/0BRqRuqnUGKEqAIu3amwsnI5nIpmzd/3DdR4qd/Xy8iVW+HY3TmgEu0rbSOjUyn6EYF55k2XYjkah1eM5nY8Dx9AVisvwVOn8NpN'}}]


In [23]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='7f9b352e-a06e-483c-a364-5795f1cc9a6a'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"body": "Hello", "subject": "Test", "recipient": "wrong@email.com"}'}, '__gemini_function_call_thought_signatures__': {'call_2433651': 'El4KXAERTTIP3ZPPc2ylW/PTDVcllWBk95AKLBmsIZiuteST8SsGPHGohuxrVtWSQRdndgRHKd0vIF5P4I/QYg41UeQllwCMm84jH18cJX8//WWirYl8FAHNafqb/Rou'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a02bc6-61de-7702-b31f-68c6eb7f7dce-0', tool_calls=[{'type': 'tool_call', 'name': 'send_email_tool', 'args': {'recipient': 'correct@email.com', 'subject': 'Corrected Subject', 'body': 'This was edited by human before sending'}, 'id': 'call_2433651'}], invalid_tool_calls=[], usage_m